# BirdCLEF 2026 — EfficientNet-B1 (Functional, no Dataset classes)

Replaces `Dataset` classes with plain functions. `batch_generator` yields `(x, y)` tensor pairs directly; soundscape rows are mixed in per batch. Everything else (model, loss, loops) is identical to the class-based version.

In [ ]:
!pip install timm librosa -q

## Imports

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import librosa
import timm

## Config

In [ ]:
CFG = {
    'train_csv':        '/kaggle/input/birdclef-2026/train.csv',
    'soundscape_csv':   '/kaggle/input/birdclef-2026/train_soundscapes_labels.csv',
    'audio_dir':        '/kaggle/input/birdclef-2026/train_audio',
    'soundscape_dir':   '/kaggle/input/birdclef-2026/train_soundscapes',
    'taxonomy_csv':     '/kaggle/input/birdclef-2026/taxonomy.csv',
    'output_dir':       '/kaggle/working',
    'sample_rate':      32000,
    'duration':         5,
    'n_mels':           128,
    'n_fft':            1024,
    'hop_length':       320,
    'fmin':             20,
    'fmax':             16000,
    'backbone':         'efficientnet_b1',
    'pretrained':       True,
    'num_classes':      234,
    'fold':             0,
    'n_folds':          5,
    'epochs':           30,
    'batch_size':       32,
    'num_workers':      4,
    'lr':               1e-3,
    'weight_decay':     1e-4,
    'min_lr':           1e-6,
    'label_smoothing':  0.05,
    'mixup_alpha':      0.4,
    'specaug_freq_mask': 24,
    'specaug_time_mask': 64,
    'seed':             42,
    'device':           'cuda' if torch.cuda.is_available() else 'cpu',
}

np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
torch.cuda.manual_seed_all(CFG['seed'])

## Labels

In [ ]:
taxonomy  = pd.read_csv(CFG['taxonomy_csv'])
SPECIES   = taxonomy['primary_label'].tolist()
LABEL2IDX = {s: i for i, s in enumerate(SPECIES)}

## Data Functions

In [ ]:
def wav_to_mel(wav, cfg):
    m = librosa.feature.melspectrogram(
        y=wav, sr=cfg['sample_rate'], n_mels=cfg['n_mels'],
        n_fft=cfg['n_fft'], hop_length=cfg['hop_length'],
        fmin=cfg['fmin'], fmax=cfg['fmax'],
    )
    m = librosa.power_to_db(m, ref=np.max)
    return ((m - m.min()) / (m.max() - m.min() + 1e-6)).astype(np.float32)

def specaugment(m, cfg):
    f  = np.random.randint(0, cfg['specaug_freq_mask'])
    f0 = np.random.randint(0, max(1, m.shape[0] - f))
    m[f0:f0+f, :] = 0
    t  = np.random.randint(0, cfg['specaug_time_mask'])
    t0 = np.random.randint(0, max(1, m.shape[1] - t))
    m[:, t0:t0+t] = 0
    return m

def load_sample(path, cfg, train=True, offset=None):
    samples = cfg['sample_rate'] * cfg['duration']
    kw = {'offset': offset, 'duration': cfg['duration']} if offset is not None else {}
    wav, _ = librosa.load(path, sr=cfg['sample_rate'], mono=True, **kw)
    if len(wav) < samples:
        wav = np.pad(wav, (0, samples - len(wav)))
    if offset is None:
        start = np.random.randint(0, max(1, len(wav) - samples)) if train else 0
        wav = wav[start:start + samples]
    mel = wav_to_mel(wav, cfg)
    if train: mel = specaugment(mel, cfg)
    return torch.tensor(mel).unsqueeze(0).repeat(3, 1, 1)

def make_label(species_str, cfg, sep=None):
    label = np.zeros(cfg['num_classes'], dtype=np.float32)
    parts = species_str.split(sep) if sep else species_str.split()
    for sp in parts:
        sp = sp.strip("[],'\" ")
        if sp in LABEL2IDX: label[LABEL2IDX[sp]] = 1.0
    return torch.tensor(label)

def collate_bird(rows, audio_dir, cfg, train=True):
    xs, ys = [], []
    for _, row in rows.iterrows():
        xs.append(load_sample(Path(audio_dir) / row['filename'], cfg, train))
        ys.append(make_label(row['primary_label'] + ' ' + str(row.get('secondary_labels', '')), cfg))
    return torch.stack(xs), torch.stack(ys)

def collate_soundscape(rows, soundscape_dir, cfg):
    xs, ys = [], []
    for _, row in rows.iterrows():
        xs.append(load_sample(Path(soundscape_dir) / row['filename'], cfg, train=True, offset=int(row['start'])))
        ys.append(make_label(str(row['primary_label']), cfg, sep=';'))
    return torch.stack(xs), torch.stack(ys)

def batch_generator(df, sc_df, cfg, train=True):
    bs = cfg['batch_size']
    if train:
        df = df.sample(frac=1).reset_index(drop=True)
    for i in range(0, len(df), bs):
        chunk = df.iloc[i:i+bs]
        # mix in soundscape rows proportionally
        sc_chunk = sc_df.sample(min(bs // 4, len(sc_df))).reset_index(drop=True) if (train and len(sc_df)) else None
        xb, yb = collate_bird(chunk, cfg['audio_dir'], cfg, train)
        if sc_chunk is not None:
            xs, ys = collate_soundscape(sc_chunk, cfg['soundscape_dir'], cfg)
            xb = torch.cat([xb, xs])
            yb = torch.cat([yb, ys])
        yield xb, yb

## Model

In [ ]:
class BirdModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(
            cfg['backbone'], pretrained=cfg['pretrained'],
            num_classes=0, global_pool='avg',
        )
        in_features = self.backbone.num_features
        self.head = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Dropout(0.3),
            nn.Linear(in_features, cfg['num_classes']),
        )

    def forward(self, x):
        return self.head(self.backbone(x))

## Loss, Mixup & Metric

In [ ]:
def mixup_batch(x, y, alpha):
    if alpha <= 0:
        return x, y
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1-lam) * x[idx], lam * y + (1-lam) * y[idx]

class BCEWithLabelSmoothing(nn.Module):
    def __init__(self, smoothing=0.05):
        super().__init__()
        self.smoothing = smoothing
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        targets = targets * (1 - self.smoothing) + self.smoothing / 2
        return self.bce(logits, targets)

def macro_roc_auc(targets, preds):
    scores = [
        roc_auc_score(targets[:, i], preds[:, i])
        for i in range(targets.shape[1]) if targets[:, i].sum() > 0
    ]
    return np.mean(scores) if scores else 0.0

## Run Training

In [ ]:
train_df      = pd.read_csv(CFG['train_csv'])
soundscape_df = pd.read_csv(CFG['soundscape_csv'])

skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
train_df['fold'] = -1
for f, (_, vi) in enumerate(skf.split(train_df, train_df['primary_label'])):
    train_df.loc[vi, 'fold'] = f

fold   = CFG['fold']
tr_df  = train_df[train_df['fold'] != fold]
val_df = train_df[train_df['fold'] == fold]

model     = BirdModel(CFG).to(CFG['device'])
criterion = BCEWithLabelSmoothing(CFG['label_smoothing'])
optimizer = optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=CFG['min_lr'])

best_auc = 0
out_path = Path(CFG['output_dir']) / f"best_fold{fold}.pt"

for epoch in range(CFG['epochs']):
    model.train()
    tr_loss = 0
    for x, y in batch_generator(tr_df, soundscape_df, CFG, train=True):
        x, y = x.to(CFG['device']), y.to(CFG['device'])
        x, y = mixup_batch(x, y, CFG['mixup_alpha'])
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item()

    model.eval()
    val_loss, all_preds, all_targets = 0, [], []
    with torch.no_grad():
        for x, y in batch_generator(val_df, pd.DataFrame(), CFG, train=False):
            x, y = x.to(CFG['device']), y.to(CFG['device'])
            logits = model(x)
            val_loss += criterion(logits, y).item()
            all_preds.append(torch.sigmoid(logits).cpu().numpy())
            all_targets.append(y.cpu().numpy())

    preds, tgt = np.concatenate(all_preds), np.concatenate(all_targets)
    auc = macro_roc_auc(tgt, preds)
    scheduler.step()
    print(f"Epoch {epoch+1:02d} | tr_loss={tr_loss:.4f} | val_loss={val_loss:.4f} | AUC={auc:.4f}")
    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), out_path)
        print(f"  -> saved (AUC={best_auc:.4f})")